# Topic 34: GPT-2, built from scratch

We write GPT-2 (the 124M model) in 47 lines of PyTorch, load OpenAI's trained weights into it,
generate text, and then fine-tune it into a small chat model (SFT). The model, the loader, the
generation loop, the chat template and the loss mask are the code of the Topic 34 slides and
lecture note.

**Runtime:** parts 1 to 5 run on a CPU. The training cell of part 6 needs a GPU:
*Runtime > Change runtime type > T4 GPU*. The whole notebook takes about 6 minutes on a T4.

**The fine-tuning here is a short version of the lecture's.** It stops after `TRAIN_SECONDS`
(4 minutes): about 600 steps of 16 examples on a T4, against 2,904 steps of 32 in the lecture's run.
The model learns to answer, but it repeats itself more than in the lecture note. Raise
`TRAIN_SECONDS` to 900 for answers closer to the note's.

In [ ]:
!pip -q install tiktoken

import json, math, random, re, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import safetensors.torch
import tiktoken
from huggingface_hub import hf_hub_download

torch.manual_seed(0); random.seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
enc = tiktoken.get_encoding("gpt2")          # GPT-2's tokenizer: byte-level BPE, 50,257 tokens
EOT = 50256                                  # <|endoftext|>
print("device:", dev)

## 1. The model

Attention, one block, the whole model. Compared with the Topic 32 decoder: no cross-attention,
pre-norm, learned positions, GELU, and the output layer reuses the token embedding (weight tying).

In [ ]:
class Config:                 # GPT-2 small, the 124M model
    vocab = 50257             # byte-level BPE tokens
    ctx = 1024                # longest input, in tokens
    d = 768                   # width of every token vector
    n_layer = 12              # decoder blocks
    n_head = 12               # heads per block, 64 dims each


class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.h = c.n_head
        self.qkv = nn.Linear(c.d, 3 * c.d)      # Q, K, V in one matmul
        self.proj = nn.Linear(c.d, c.d)

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=2)
        hd = d // self.h                                    # 64 dims per head
        q, k, v = (t.view(B, T, self.h, hd).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(hd)
        past = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        scores = scores.masked_fill(~past, float("-inf"))   # causal mask
        y = scores.softmax(dim=-1) @ v
        return self.proj(y.transpose(1, 2).reshape(B, T, d))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.d), nn.LayerNorm(c.d)
        self.attn = CausalSelfAttention(c)
        self.mlp = nn.Sequential(nn.Linear(c.d, 4 * c.d),
                                 nn.GELU(approximate="tanh"),
                                 nn.Linear(4 * c.d, c.d))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm: normalize, then add
        x = x + self.mlp(self.ln2(x))
        return x


class GPT2(nn.Module):
    def __init__(self, c=Config):
        super().__init__()
        self.wte = nn.Embedding(c.vocab, c.d)    # token embedding
        self.wpe = nn.Embedding(c.ctx, c.d)      # learned position embedding
        self.blocks = nn.ModuleList(Block(c) for _ in range(c.n_layer))
        self.ln_f = nn.LayerNorm(c.d)

    def forward(self, idx):                      # idx: (B, T) token ids
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.wte(idx) + self.wpe(pos)
        for block in self.blocks:
            x = block(x)
        return self.ln_f(x) @ self.wte.weight.T  # (B, T, vocab) logits

## 2. OpenAI's weights

The weights are public. Only the names differ from ours (`h.0.ln_1` is our `blocks.0.ln1`), and
OpenAI stored every Linear weight as (in, out), so those are transposed.

In [ ]:
def load_openai_weights(model, path):
    sd = safetensors.torch.load_file(path)       # 548 MB, from Hugging Face
    own = model.state_dict()
    for k, v in sd.items():
        if k.endswith((".attn.bias", ".attn.masked_bias")):
            continue                             # a stored mask, not a weight
        name = re.sub(r"^h\.", "blocks.", k)
        for a, b in [("ln_1", "ln1"), ("ln_2", "ln2"), ("c_attn", "qkv"),
                     ("attn.c_proj", "attn.proj"),
                     ("c_fc", "0"), ("mlp.c_proj", "mlp.2")]:
            name = name.replace(a, b)
        if v.ndim == 2 and not name.startswith(("wte", "wpe")):
            v = v.T                              # OpenAI stored (in, out)
        own[name].copy_(v)
    return model


path = hf_hub_download("openai-community/gpt2", "model.safetensors")     # 548 MB
model = load_openai_weights(GPT2(), path).to(dev).eval()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
for name, w in list(model.state_dict().items())[:6]:                     # names come from the attribute names
    print(f"  {name:28s} {tuple(w.shape)}")

**Check:** the same logits as the Hugging Face implementation?

In [ ]:
from transformers import GPT2LMHeadModel

hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to(dev).eval()
ids = torch.tensor([enc.encode("The decoder writes one token at a time, and every token it writes becomes part of its input.")], device=dev)
with torch.no_grad():
    a, b = model(ids), hf(ids).logits
print(f"largest difference in any logit: {(a - b).abs().max().item():.1e}   (logits are as large as {a.abs().max().item():.0f})")
print("same top token at every position:", bool((a.argmax(-1) == b.argmax(-1)).all()))
del hf

## 3. The tokenizer

A space belongs to the token after it, nothing is ever unknown, and text unlike the training data
costs more tokens. `[..]` marks one token and `_` a space.

In [ ]:
def pieces(text):
    out = []
    for i in enc.encode(text):
        b = enc.decode_single_token_bytes(i)
        try:
            out.append("[" + b.decode("utf-8").replace(" ", "_").replace("\n", "\\n") + "]")
        except UnicodeDecodeError:               # part of a character: show the raw bytes
            out.append("[" + "".join(f"<{x:02X}>" for x in b) + "]")
    return "".join(out)


for text in ["Hello world", "Hello World", "Hello        world", "GPT-2 was released in 2019.",
             "In 2026, 12345 people came.", "¿Dónde está la biblioteca?", "안녕하세요"]:
    print(f"{len(enc.encode(text)):3d} tokens  {text!r:32s} {pieces(text)}")
print("ids of 'world', ' world', 'World', ' World':", [enc.encode(w)[0] for w in ["world", " world", "World", " World"]])

## 4. Generation

The loop: run the model on everything so far, keep the last position's logits, pick a token,
append it, repeat.

In [ ]:
@torch.no_grad()
def generate(model, ids, n, temperature=1.0, top_k=None, greedy=False):
    # ids: the prompt as token ids, shape (1, T)
    for _ in range(n):                            # one new token per pass
        logits = model(ids[:, -1024:])[:, -1, :]  # last position's logits
        if greedy:
            nxt = logits.argmax(dim=-1, keepdim=True)  # top token
        else:
            logits = logits / temperature  # T<1 sharper, T>1 flatter
            if top_k:                      # keep the k largest
                kth = logits.topk(top_k).values[:, [-1]]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            nxt = torch.multinomial(logits.softmax(dim=-1), 1)  # draw one
        ids = torch.cat([ids, nxt], dim=1)        # append: (1,T) -> (1,T+1)
        if nxt.item() == EOT:                     # <|endoftext|>: stop
            break
    return ids                                    # prompt + new tokens


def write(prompt, n=60, seed=0, **how):
    torch.manual_seed(seed)
    ids = torch.tensor([enc.encode(prompt, allowed_special={"<|endoftext|>"})], device=dev)
    out = generate(model, ids, n, **how)
    return enc.decode([t for t in out[0, ids.size(1):].tolist() if t != EOT])


story = "Once upon a time, there was a little robot who"
print("GREEDY:\n", write(story, greedy=True))
print("\nT = 0.8, top-k = 40:\n", write(story, seed=1, temperature=0.8, top_k=40))
print("\nT = 2.0:\n", write(story, n=40, seed=1, temperature=2.0))

What does GPT-2 think comes after *The capital of France is*? Try other prompts and temperatures.

In [ ]:
@torch.no_grad()
def next_tokens(prompt, k=6, T=1.0):
    ids = torch.tensor([enc.encode(prompt)], device=dev)
    p = (model(ids)[0, -1] / T).softmax(-1)
    top = p.topk(k)
    for v, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  {enc.decode([i])!r:12s} {100 * v:5.1f}%")
    print(f"  the other {enc.n_vocab - k:,} tokens: {100 * (1 - top.values.sum().item()):.0f}%")


next_tokens("The capital of France is")

## 5. Prompting a pretrained model

Few-shot prompting works when the model has the knowledge (Paris) and fails when it does not
(Spanish). With a chat transcript and no examples, it just continues the transcript.

In [ ]:
print(repr(write("Q: What is the capital of Italy?\nA: Rome\nQ: What is the capital of Japan?\nA: Tokyo\nQ: What is the capital of France?\nA:", n=6, greedy=True)))
print(repr(write("English: good morning\nSpanish: buenos días\nEnglish: thank you\nSpanish: gracias\nEnglish: good night\nSpanish:", n=6, greedy=True)))
print(write("User: What is the capital of France?\nAssistant:", n=40, greedy=True))

## 6. Making it chat: supervised fine-tuning (SFT)

Each (instruction, answer) pair of Alpaca becomes one document in a chat template. **The loss is
computed on the answer only**: every prompt token gets the label -100, which `cross_entropy` ignores.

The two cells below still run on a CPU; the training cell after them needs a GPU.

In [ ]:
def chat(user, assistant=None):
    prompt = f"<|user|>\n{user}\n<|assistant|>\n"
    if assistant is None:
        return prompt                                  # at inference: the model continues here
    return prompt, assistant + "<|endoftext|>"


def encode(user, assistant):
    prompt, answer = chat(user, assistant)
    p = enc.encode(prompt)
    a = enc.encode(answer, allowed_special={"<|endoftext|>"})
    ids = p + a
    labels = [-100] * len(p) + a                       # -100: no loss on the prompt
    return ids[:-1], labels[1:]                        # predict token t+1 from tokens <= t


MAX_LEN = 256
rows = json.load(open(hf_hub_download("yahma/alpaca-cleaned", "alpaca_data_cleaned.json", repo_type="dataset")))
random.shuffle(rows)
data = []
for r in rows:
    user = r["instruction"] + ("\n\n" + r["input"] if r["input"].strip() else "")
    x, y = encode(user, r["output"])
    if len(x) <= MAX_LEN:
        data.append((x, y))
val, train = data[:300], data[300:]
print(f"{len(rows):,} pairs, {len(data):,} fit in {MAX_LEN} tokens")
x, y = train[0]
print("input :", enc.decode(x)[:90].replace("\n", "\\n"), "...")
print("labels:", y[:12], "...   (-100 = prompt token, not graded)")

In [ ]:
def batchify(items):
    T = max(len(x) for x, _ in items)
    X = torch.full((len(items), T), EOT, dtype=torch.long)
    Y = torch.full((len(items), T), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X.to(dev), Y.to(dev)


def answer(question, n=80, **how):
    model.eval()
    out = write(chat(question), n=n, greedy=not how, **how)
    model.train()
    return out


@torch.no_grad()
def val_loss():
    model.eval(); tot = cnt = 0
    for i in range(0, len(val), 50):
        X, Y = batchify(val[i:i + 50])
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lg = model(X)
        tot += F.cross_entropy(lg.float().view(-1, lg.size(-1)), Y.view(-1), ignore_index=-100, reduction="sum").item()
        cnt += (Y != -100).sum().item()
    model.train()
    return tot / cnt


QUESTIONS = ["What is the capital of France?", "Write a haiku about the ocean.",
             "Explain what a neural network is in one sentence.", "What is 12 plus 7?"]
before = {q: answer(q, n=40) for q in QUESTIONS}
print("BEFORE fine-tuning:\n", before[QUESTIONS[0]])
print("validation loss before:", round(val_loss(), 3))

The training step is the ordinary one, and all 124 million weights move. Compared with the slides it
uses float16 with a `GradScaler` (a T4 has no fast bfloat16), a batch of 16, and a short warm-up with
no cosine decay, because it stops after `TRAIN_SECONDS` rather than after a fixed number of steps.

In [ ]:
assert dev == "cuda", "fine-tuning needs a GPU: Runtime > Change runtime type > T4 GPU"
TRAIN_SECONDS = 240
BATCH, LR, WARMUP = 16, 1e-4, 50

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler("cuda")                # float16 on the GPU: faster, less memory
model.train()
t0, step, next_report = time.time(), 0, 30
while time.time() - t0 < TRAIN_SECONDS:
    X, Y = batchify(random.sample(train, BATCH))
    for group in opt.param_groups:
        group["lr"] = LR * min(1.0, (step + 1) / WARMUP)                       # short warm-up
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(X)                                                      # (B, T, vocab)
    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), Y.view(-1),
                           ignore_index=-100)                                  # answer tokens only
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad()
    step += 1
    if time.time() - t0 >= next_report:
        print(f"step {step:5d}  train loss {loss.item():.3f}  val loss {val_loss():.3f}  ({time.time() - t0:.0f} s)")
        next_report += 30
print(f"trained {step} steps of {BATCH} in {time.time() - t0:.0f} s on {torch.cuda.get_device_name()}")

## 7. Before and after

Before fine-tuning the model repeats the question. After it, the model answers. With only a few
hundred steps it often does not stop cleanly yet (the lecture's longer run does).

In [ ]:
for q in QUESTIONS:
    print("QUESTION:", q)
    print("  before:", before[q].replace("\n", " / ")[:110])
    print("  after :", answer(q).replace("\n", " / "))
    print()

A second turn: the first answer is pasted into the context, then the next question. And your own
question. Check the facts and the arithmetic yourself: SFT taught the format, not knowledge.

In [ ]:
first = answer("What is the capital of France?")
context = chat("What is the capital of France?") + first + "<|endoftext|>" + chat("What is its population?")
model.eval()
print(context + write(context, n=60, greedy=True))
print()
print(answer("Give me three tips for staying healthy."))        # change the question and run again